# Model Comparison: Predicting Midfielder Quality Changes After Transfer

Six OLS regression models of increasing complexity, each fitted independently for 17 player qualities. For each model we report the **top 5 best-predicted qualities** on the test set.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error

# ── Load & prepare data (same as notebook 01) ────────────────────────────

DATA_DIR = "../../../../thesis_data/processed_data/thesis_model_dataset/active/"
df = pd.read_parquet(DATA_DIR + "within_league_transfers_v5.parquet")
mf = df[df["from_position"] == "Midfielder"].copy()

QUALITIES = [
    "Active defence", "Aerial threat", "Box threat", "Composure",
    "Defensive heading", "Dribbling", "Effectiveness", "Finishing",
    "Hold-up play", "Intelligent defence", "Involvement",
    "Passing quality", "Pressing", "Progression",
    "Providing teammates", "Run quality", "Winning duels",
]

TEAM_Q = ["DEFENCE", "DEFENSIVE_TRANSITION", "ATTACKING_TRANSITION",
          "ATTACK", "PENETRATION", "CHANCE_CREATION", "OUTCOME"]

from_pq = [f"from_{q}" for q in QUALITIES]
to_pq   = [f"to_{q}" for q in QUALITIES]
from_tq = [f"from_q_proj_{q}" for q in TEAM_Q]
to_tq   = [f"to_q_{q}" for q in TEAM_Q]

# Deltas
for q in QUALITIES:
    mf[f"delta_{q}"] = mf[f"to_{q}"] - mf[f"from_{q}"]
for q in TEAM_Q:
    mf[f"delta_team_{q}"] = mf[f"to_q_{q}"] - mf[f"from_q_proj_{q}"]
mf["delta_Minutes"] = mf["to_Minutes"] - mf["from_Minutes"]

delta_tq = [f"delta_team_{q}" for q in TEAM_Q]
delta_pq = [f"delta_{q}" for q in QUALITIES]

# Clean & split
all_cols = (from_pq + to_pq + from_tq + to_tq + delta_tq + delta_pq
            + ["player_season_age", "delta_Minutes", "from_season"])
mf_clean = mf[all_cols].dropna()

train = mf_clean[mf_clean["from_season"] <= 2023]
test  = mf_clean[mf_clean["from_season"] == 2024]

print(f"Train: {len(train):,} | Test: {len(test):,}")

Train: 4,383 | Test: 465


In [2]:
def run_model(train, test, features_fn, target_fn, model_name):
    """
    Run 17 OLS regressions (one per quality) and return a results DataFrame.
    
    Parameters
    ----------
    features_fn : callable(quality_name) -> list of column names
    target_fn   : callable(quality_name) -> column name
    """
    rows = []
    for q in QUALITIES:
        feat_cols = features_fn(q)
        target_col = target_fn(q)

        X_train = sm.add_constant(train[feat_cols])
        X_test  = sm.add_constant(test[feat_cols])
        y_train = train[target_col]
        y_test  = test[target_col]

        model = sm.OLS(y_train, X_train).fit()
        y_pred = model.predict(X_test)

        r2_test = 1 - ((y_test - y_pred) ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum()

        rows.append({
            "Quality": q,
            "R2_train": model.rsquared,
            "R2_adj_train": model.rsquared_adj,
            "R2_test": r2_test,
            "MAE_test": mean_absolute_error(y_test, y_pred),
            "F_pvalue": model.f_pvalue,
            "N_features": len(feat_cols),
        })

    results = pd.DataFrame(rows)
    results["Model"] = model_name
    return results

---
## Model 1: Naive Baseline

$$\text{to\_Q}_i = \alpha + \beta \cdot \text{from\_Q}_i$$

The simplest hypothesis: a player's post-transfer quality is a linear function of only that same quality before the transfer. One univariate regression per quality.

In [3]:
m1 = run_model(
    train, test,
    features_fn=lambda q: [f"from_{q}"],
    target_fn=lambda q: f"to_{q}",
    model_name="1. Naive",
)
m1.sort_values("R2_test", ascending=False).head(5).round(4)

,Quality,R2_train,R2_adj_train,R2_test,MAE_test,F_pvalue,N_features,Model
11,Passing quality,0.4703,0.4701,0.4472,0.4087,0.0,1,1. Naive
13,Progression,0.4311,0.4310,0.3796,0.4920,0.0,1,1. Naive
15,Run quality,0.4699,0.4698,0.3682,0.3157,0.0,1,1. Naive
14,Providing teammates,0.3860,0.3858,0.3596,0.4201,0.0,1,1. Naive
4,Defensive heading,0.3742,0.3741,0.3587,0.4913,0.0,1,1. Naive


---
## Model 2: All Pre-Transfer Qualities

$$\text{to\_Q}_i = \alpha + \sum_{j=1}^{17} \beta_j \cdot \text{from\_Q}_j$$

All 17 pre-transfer qualities predict each post-transfer quality. Captures cross-quality effects (e.g., does pre-transfer Pressing help predict post-transfer Involvement?).

In [4]:
m2 = run_model(
    train, test,
    features_fn=lambda q: from_pq,
    target_fn=lambda q: f"to_{q}",
    model_name="2. All Pre-Q",
)
m2.sort_values("R2_test", ascending=False).head(5).round(4)

,Quality,R2_train,R2_adj_train,R2_test,MAE_test,F_pvalue,N_features,Model
11,Passing quality,0.4916,0.4896,0.4641,0.4024,0.0,17,2. All Pre-Q
13,Progression,0.4520,0.4499,0.4119,0.4745,0.0,17,2. All Pre-Q
4,Defensive heading,0.4231,0.4208,0.4036,0.4753,0.0,17,2. All Pre-Q
15,Run quality,0.4955,0.4935,0.4004,0.3103,0.0,17,2. All Pre-Q
14,Providing teammates,0.4380,0.4358,0.4001,0.4045,0.0,17,2. All Pre-Q


---
## Model 3: Pre-Qualities + Team Styles

$$\text{to\_Q}_i = \alpha + \sum_{j=1}^{17} \beta_j \cdot \text{from\_Q}_j + \sum_{k=1}^{7} \gamma_k \cdot \text{from\_team}_k + \sum_{k=1}^{7} \delta_k \cdot \text{to\_team}_k$$

Adds the tactical style of both the origin team (projected) and destination team. Tests whether team context explains variance beyond the player's own qualities.

In [5]:
m3 = run_model(
    train, test,
    features_fn=lambda q: from_pq + from_tq + to_tq,
    target_fn=lambda q: f"to_{q}",
    model_name="3. Pre-Q + Teams",
)
m3.sort_values("R2_test", ascending=False).head(5).round(4)

,Quality,R2_train,R2_adj_train,R2_test,MAE_test,F_pvalue,N_features,Model
11,Passing quality,0.5454,0.5422,0.5210,0.3876,0.0,31,3. Pre-Q + Teams
10,Involvement,0.4658,0.4620,0.4935,0.3256,0.0,31,3. Pre-Q + Teams
14,Providing teammates,0.4654,0.4616,0.4384,0.3908,0.0,31,3. Pre-Q + Teams
13,Progression,0.4777,0.4740,0.4360,0.4709,0.0,31,3. Pre-Q + Teams
15,Run quality,0.5162,0.5128,0.4210,0.3061,0.0,31,3. Pre-Q + Teams


---
## Model 4a: Delta Team Only

$$\Delta PQ_i = \alpha + \sum_{k=1}^{7} \gamma_k \cdot \Delta TQ_k$$

Can the change in tactical environment alone explain how player qualities change? No player baseline information — purely team-driven.

In [6]:
m4a = run_model(
    train, test,
    features_fn=lambda q: delta_tq,
    target_fn=lambda q: f"delta_{q}",
    model_name="4a. Delta Team",
)
m4a.sort_values("R2_test", ascending=False).head(5).round(4)

,Quality,R2_train,R2_adj_train,R2_test,MAE_test,F_pvalue,N_features,Model
10,Involvement,0.1733,0.1720,0.1659,0.3669,0.0,7,4a. Delta Team
3,Composure,0.1033,0.1019,0.1242,0.4919,0.0,7,4a. Delta Team
6,Effectiveness,0.0747,0.0732,0.1072,0.2868,0.0,7,4a. Delta Team
11,Passing quality,0.0826,0.0811,0.0880,0.4319,0.0,7,4a. Delta Team
12,Pressing,0.0454,0.0438,0.0599,0.5137,0.0,7,4a. Delta Team


---
## Model 4b: Pre-Qualities + Delta Team

$$\Delta PQ_i = \alpha + \sum_{j=1}^{17} \beta_j \cdot \text{from\_Q}_j + \sum_{k=1}^{7} \gamma_k \cdot \Delta TQ_k$$

The key model. Combines the player's baseline with the change in tactical context. Negative coefficients on `from_Q_j` capture regression to the mean; delta team features capture environmental push.

In [7]:
m4b = run_model(
    train, test,
    features_fn=lambda q: from_pq + delta_tq,
    target_fn=lambda q: f"delta_{q}",
    model_name="4b. Pre-Q + Delta Team",
)
m4b.sort_values("R2_test", ascending=False).head(5).round(4)

,Quality,R2_train,R2_adj_train,R2_test,MAE_test,F_pvalue,N_features,Model
3,Composure,0.4315,0.4283,0.5131,0.3713,0.0,24,4b. Pre-Q + Delta Team
7,Finishing,0.4543,0.4513,0.4432,0.5244,0.0,24,4b. Pre-Q + Delta Team
6,Effectiveness,0.3570,0.3534,0.4381,0.2361,0.0,24,4b. Pre-Q + Delta Team
16,Winning duels,0.3671,0.3636,0.4143,0.3586,0.0,24,4b. Pre-Q + Delta Team
8,Hold-up play,0.3620,0.3585,0.4007,0.2552,0.0,24,4b. Pre-Q + Delta Team


---
## Model 4c: + Age & Minutes Controls

$$\Delta PQ_i = \alpha + \sum_{j=1}^{17} \beta_j \cdot \text{from\_Q}_j + \sum_{k=1}^{7} \gamma_k \cdot \Delta TQ_k + \delta \cdot \text{age} + \varepsilon \cdot \Delta\text{minutes}$$

Adds age (younger players improve, older decline) and change in playing time (more/fewer minutes affect stats mechanically).

In [8]:
m4c = run_model(
    train, test,
    features_fn=lambda q: from_pq + delta_tq + ["player_season_age", "delta_Minutes"],
    target_fn=lambda q: f"delta_{q}",
    model_name="4c. + Controls",
)
m4c.sort_values("R2_test", ascending=False).head(5).round(4)

,Quality,R2_train,R2_adj_train,R2_test,MAE_test,F_pvalue,N_features,Model
3,Composure,0.4333,0.4299,0.5111,0.3716,0.0,26,4c. + Controls
7,Finishing,0.4553,0.4521,0.4412,0.5237,0.0,26,4c. + Controls
6,Effectiveness,0.3587,0.3549,0.4382,0.2356,0.0,26,4c. + Controls
16,Winning duels,0.3680,0.3642,0.4095,0.3603,0.0,26,4c. + Controls
8,Hold-up play,0.3625,0.3587,0.3981,0.2556,0.0,26,4c. + Controls


---
## Results Summary

### Top 5 best-predicted qualities per model

In [9]:
all_results = pd.concat([m1, m2, m3, m4a, m4b, m4c], ignore_index=True)

# Top 5 per model
top5 = (all_results
        .sort_values(["Model", "R2_test"], ascending=[True, False])
        .groupby("Model")
        .head(5))

display_cols = ["Model", "Quality", "R2_test", "R2_adj_train", "MAE_test", "F_pvalue"]

for model_name, group in top5.groupby("Model", sort=False):
    print(f"\n{'='*70}")
    print(f"  {model_name}")
    print(f"{'='*70}")
    print(group[display_cols].to_string(index=False, float_format="{:.4f}".format))


  1. Naive
   Model             Quality  R2_test  R2_adj_train  MAE_test  F_pvalue
1. Naive     Passing quality   0.4472        0.4701    0.4087    0.0000
1. Naive         Progression   0.3796        0.4310    0.4920    0.0000
1. Naive         Run quality   0.3682        0.4698    0.3157    0.0000
1. Naive Providing teammates   0.3596        0.3858    0.4201    0.0000
1. Naive   Defensive heading   0.3587        0.3741    0.4913    0.0000

  2. All Pre-Q
       Model             Quality  R2_test  R2_adj_train  MAE_test  F_pvalue
2. All Pre-Q     Passing quality   0.4641        0.4896    0.4024    0.0000
2. All Pre-Q         Progression   0.4119        0.4499    0.4745    0.0000
2. All Pre-Q   Defensive heading   0.4036        0.4208    0.4753    0.0000
2. All Pre-Q         Run quality   0.4004        0.4935    0.3103    0.0000
2. All Pre-Q Providing teammates   0.4001        0.4358    0.4045    0.0000

  3. Pre-Q + Teams
           Model             Quality  R2_test  R2_adj_train  MAE

### Full results (all 17 qualities x 6 models)

In [10]:
# Pivot R2_test
r2_pivot = all_results.pivot(index="Quality", columns="Model", values="R2_test").round(4)

# Pivot F_pvalue
pval_pivot = all_results.pivot(index="Quality", columns="Model", values="F_pvalue")

# Combined view: R2 with significance markers
def fmt_cell(r2, pval):
    stars = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    return f"{r2:.4f}{stars}"

combined = pd.DataFrame(index=r2_pivot.index, columns=r2_pivot.columns)
for col in r2_pivot.columns:
    for idx in r2_pivot.index:
        combined.loc[idx, col] = fmt_cell(r2_pivot.loc[idx, col], pval_pivot.loc[idx, col])

# Best model per quality (highest R2 with p < 0.05)
valid = all_results[all_results["F_pvalue"] < 0.05]
best = valid.loc[valid.groupby("Quality")["R2_test"].idxmax()][["Quality", "Model", "R2_test", "MAE_test", "F_pvalue"]]
best = best.sort_values("R2_test", ascending=False).reset_index(drop=True)

print("R2 test by Quality x Model (*** p<0.001, ** p<0.01, * p<0.05)")
print("=" * 120)
print(combined.to_string())
print()
print("Best model per quality (p < 0.05)")
print("=" * 80)
print(best.to_string(index=False, float_format="{:.4f}".format))


R2 test by Quality x Model (*** p<0.001, ** p<0.01, * p<0.05)
Model                 1. Naive 2. All Pre-Q 3. Pre-Q + Teams 4a. Delta Team 4b. Pre-Q + Delta Team 4c. + Controls
Quality                                                                                                          
Active defence       0.3013***    0.3480***        0.3514***     -0.0021***              0.3028***      0.3005***
Aerial threat        0.1852***    0.2499***        0.2688***      0.0298***              0.3252***      0.3278***
Box threat           0.3435***    0.3622***        0.3853***      0.0364***              0.2798***      0.2837***
Composure            0.0290***    0.0759***        0.1556***      0.1242***              0.5131***      0.5111***
Defensive heading    0.3587***    0.4036***        0.4030***      0.0002***              0.2329***      0.2345***
Dribbling            0.1896***    0.2053***        0.2176***     -0.0027***              0.3115***      0.3080***
Effectiveness        0.147

---
## Regression to Mean Warning

Models 4b and 4c predict `delta_Q = to_Q - from_Q` and include `from_Q` as a feature. This creates an artificial R² inflation:
the model learns that extreme pre-transfer values regress toward the mean (coef ≈ -1), which is trivially predictable
but not a useful tactical insight.

The analysis below decomposes each quality's R² into what comes from **mean reversion** vs. **actual tactical change**.

In [11]:
# Regression to Mean decomposition for Model 4b
rtm_rows = []
for q in QUALITIES:
    target = f"delta_{q}"
    from_col = f"from_{q}"

    # Full 4b
    X_full = sm.add_constant(train[from_pq + delta_tq])
    X_full_t = sm.add_constant(test[from_pq + delta_tq])
    m_full = sm.OLS(train[target], X_full).fit()
    p_full = m_full.predict(X_full_t)
    r2_full = 1 - ((test[target] - p_full)**2).sum() / ((test[target] - test[target].mean())**2).sum()

    # Only from_Q_i (pure mean reversion)
    X_naive = sm.add_constant(train[[from_col]])
    X_naive_t = sm.add_constant(test[[from_col]])
    m_naive = sm.OLS(train[target], X_naive).fit()
    p_naive = m_naive.predict(X_naive_t)
    r2_naive = 1 - ((test[target] - p_naive)**2).sum() / ((test[target] - test[target].mean())**2).sum()

    # Only delta team (pure tactical)
    X_tq = sm.add_constant(train[delta_tq])
    X_tq_t = sm.add_constant(test[delta_tq])
    m_tq = sm.OLS(train[target], X_tq).fit()
    p_tq = m_tq.predict(X_tq_t)
    r2_tq = 1 - ((test[target] - p_tq)**2).sum() / ((test[target] - test[target].mean())**2).sum()

    rtm_rows.append({
        "Quality": q,
        "corr(from,delta)": train[from_col].corr(train[target]),
        "own_coef": m_full.params[from_col],
        "R2_4b_full": r2_full,
        "R2_mean_reversion": r2_naive,
        "R2_tactical_only": r2_tq,
        "R2_inflation": r2_full - r2_tq,
    })

rtm = pd.DataFrame(rtm_rows).sort_values("corr(from,delta)")
print("Regression to Mean Decomposition — Model 4b")
print("=" * 120)
print(rtm.to_string(index=False, float_format="{:.4f}".format))
print()
print("Conclusion: Models 4b/4c R² is dominated by mean reversion, not tactical insight.")
print("A fair comparison requires measuring all models on the same target (to_Q).")

Regression to Mean Decomposition — Model 4b
            Quality  corr(from,delta)  own_coef  R2_4b_full  R2_mean_reversion  R2_tactical_only  R2_inflation
          Finishing           -0.6465   -1.0073      0.4432             0.4235            0.0069        0.4364
          Composure           -0.6031   -0.7968      0.5131             0.4563            0.1242        0.3888
       Hold-up play           -0.5498   -0.7119      0.4007             0.3454            0.0318        0.3689
      Winning duels           -0.5462   -0.8630      0.4143             0.3375           -0.0066        0.4209
      Effectiveness           -0.5048   -0.7684      0.4381             0.3335            0.1072        0.3309
          Dribbling           -0.4950   -0.6249      0.3115             0.2904           -0.0027        0.3142
           Pressing           -0.4808   -0.5197      0.3045             0.2234            0.0599        0.2446
      Aerial threat           -0.4800   -0.7317      0.3252         